# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [34]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [35]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['XAUNKVRWOK', 'OAVXIRKFMW'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[24,  1, 21, 14, 11, 22, 18, 23, 15, 11],
       [15,  1, 22, 24,  9, 18, 11,  6, 13, 23]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 11, 15, 23, 18, 22, 11, 14, 21,  1],
       [ 0, 23, 13,  6, 11, 18,  9, 24, 22,  1]])>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[11, 15, 23, 18, 22, 11, 14, 21,  1, 24],
       [23, 13,  6, 11, 18,  9, 24, 22,  1, 15]])>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [40]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    @tf.function
    def call(self, enc_ids, dec_ids):
        enc_emb = self.embed_layer(enc_ids)
        enc_out, enc_state = self.encoder(enc_emb)

        dec_emb = self.embed_layer(dec_ids)

        dec_len = tf.shape(dec_ids)[1]
        state = enc_state

        # 用 TensorArray 替代 list
        outputs = tf.TensorArray(dtype=tf.float32, size=dec_len)

        for t in tf.range(dec_len):
            x_t = dec_emb[:, t, :]

            out, state = self.decoder_cell(x_t, state)

            # attention
            score = tf.matmul(self.dense_attn(enc_out), tf.expand_dims(out, -1))
            score = tf.squeeze(score, -1)
            attn_weights = tf.nn.softmax(score, axis=1)

            context = tf.reduce_sum(
                enc_out * tf.expand_dims(attn_weights, -1), axis=1
            )

            concat = tf.concat([out, context], axis=-1)
            logit = self.dense(concat)

            # 写入 TensorArray
            outputs = outputs.write(t, logit)

        # stack
        logits = outputs.stack()          # [T, B, V]
        logits = tf.transpose(logits, [1, 0, 2])  # [B, T, V]

        return logits
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        # x: [b_sz]

        x = self.embed_layer(x)  # [b, emb]

        # RNN step
        out, state = self.decoder_cell(x, state)

        # ===== Attention =====
        score = tf.matmul(self.dense_attn(enc_out), tf.expand_dims(out, -1))
        score = tf.squeeze(score, -1)

        attn_weights = tf.nn.softmax(score, axis=1)

        context = tf.reduce_sum(
            enc_out * tf.expand_dims(attn_weights, -1), axis=1
        )

        concat = tf.concat([out, context], axis=-1)

        logits = self.dense(concat)

        return logits, state

# Loss函数以及训练逻辑

In [41]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [42]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.3043377
step 500 : loss 1.5821545
step 1000 : loss 0.06337632
step 1500 : loss 0.04400796


<tf.Tensor: shape=(), dtype=float32, numpy=0.012325464>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [49]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]

        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state

        outputs = []

        for _ in range(steps):
            logits, state = model.get_next_token(cur_token, state, enc_out)
            cur_token = tf.argmax(logits, axis=-1, output_type=tf.int32)
            outputs.append(cur_token)

        return tf.stack(outputs, axis=1)

    # 测试
    batched_examples, enc_x, dec_x, y = get_batch(2, 10)
    enc_out, init_state = model.encode(enc_x)

    pred = decode(init_state, 10, enc_out)

    print("input:", enc_x.numpy())
    print("pred :", pred.numpy())
    print("true :", y.numpy())
    return enc_x.numpy(), pred.numpy(), y.numpy()

def is_reverse(inp, pred, true):
    return (pred == true).all()
    
print([is_reverse(*item) for item in zip(*sequence_reversal())])
print(list(zip(*sequence_reversal())))

input: [[ 5  1  9 19  2 20 24 14 20  5]
 [16 24 12  3  3  5 19  4 12  6]]
pred : [[ 5 20 14 24 20  2 19  9  1  5]
 [ 6 12  4 19  5  3  3 12 24 16]]
true : [[ 5 20 14 24 20  2 19  9  1  5]
 [ 6 12  4 19  5  3  3 12 24 16]]
[True, True]
input: [[ 1 13 24  1  6 16 11 23 24 21]
 [10  9 14 17  8 21  5 16 25 19]]
pred : [[21 24 23 11 16  6  1 24 13  1]
 [19 25 16  5 21  8 17 14  9 10]]
true : [[21 24 23 11 16  6  1 24 13  1]
 [19 25 16  5 21  8 17 14  9 10]]
[(array([ 1, 13, 24,  1,  6, 16, 11, 23, 24, 21]), array([21, 24, 23, 11, 16,  6,  1, 24, 13,  1]), array([21, 24, 23, 11, 16,  6,  1, 24, 13,  1])), (array([10,  9, 14, 17,  8, 21,  5, 16, 25, 19]), array([19, 25, 16,  5, 21,  8, 17, 14,  9, 10]), array([19, 25, 16,  5, 21,  8, 17, 14,  9, 10]))]
